In [5]:
import requests
from bs4 import BeautifulSoup
import json

# Function to get author information from Google Scholar
def get_author_info(author_url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

    response = requests.get(author_url, headers=headers)
    if response.status_code != 200:
        print("Failed to retrieve page.")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract author's full name
    try:
        name = soup.find('div', id='gsc_prf_in').text.strip()
    except AttributeError:
        name = None

    # Extract author's affiliation (this includes country info if available)
    try:
        affiliation = soup.find('div', class_='gsc_prf_il').text.strip()
    except AttributeError:
        affiliation = None

    # Extract total citations
    try:
        citations = soup.find_all('td', class_='gsc_rsb_std')[0].text.strip()
    except (AttributeError, IndexError):
        citations = None

    # Extract h-index
    try:
        h_index = soup.find_all('td', class_='gsc_rsb_std')[2].text.strip()
    except (AttributeError, IndexError):
        h_index = None

    # Extract co-authors
    try:
        co_authors = [co_author.text for co_author in soup.find_all('span', class_='gsc_rsb_a_desc')]
    except AttributeError:
        co_authors = []

    # Extract list of publications
    publications = []
    try:
        pub_list = soup.find_all('tr', class_='gsc_a_tr')
        for pub in pub_list:
            title = pub.find('a', class_='gsc_a_at').text
            authors_info = pub.find('div', class_='gs_gray').text  # Authors
            source_info = pub.find_all('div', class_='gs_gray')[1].text  # Journal or conference name
            year = pub.find('span', class_='gsc_a_h').text  # Year of publication
            citations = pub.find('a', class_='gsc_a_ac').text  # Citations count (if available)

            publications.append({
                'Title': title,
                'Authors': authors_info,
                'Source': source_info,
                'Year': year,
                'Citations': citations
            })
    except AttributeError:
        pass

    author_info = {
        'Name': name,
        'Affiliation': affiliation,
        'Total Citations': citations,
        'H-index': h_index,
        'Co-authors': co_authors,
        'Publications': publications
    }

    return author_info

# Save data as JSON file
def save_as_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"Data saved to {filename}")

# Example usage
author_url = "https://scholar.google.com/citations?user=WLN3QrAAAAAJ&hl=en"  # Replace with the actual author profile URL
author_data = get_author_info(author_url)

if author_data:
    save_as_json(author_data, 'author_info.json')

Data saved to author_info.json
